# Question 3

## (a) Loading the Brown corpus

We load the tagged sentences from the "news" category of the Brown corpus, simplify complex tags (taking the prefix before the first '+' or '-'), and split the last 10% of sentences as the test set.

In [1]:
from tagger import load_brown_news

train_sents, test_sents = load_brown_news()

print(f"Training sentences: {len(train_sents)}")
print(f"Test sentences:     {len(test_sents)}")
print(f"Example sentence:   {train_sents[0][:5]} ...")

Training sentences: 4160
Test sentences:     463
Example sentence:   [('The', 'AT'), ('Fulton', 'NP'), ('County', 'NN'), ('Grand', 'JJ'), ('Jury', 'NN')] ...


## (b) Most likely tag baseline

For each word in the training set we compute its most likely tag (MLE: most frequent tag for that word). Unknown words are assigned the tag "NN". We then evaluate error rates on the test set.

### (i) Training

We compute the most likely tag for each known word using maximum likelihood estimation.

In [2]:
from tagger import train_most_likely_tag

word_to_tag = train_most_likely_tag(train_sents)

### (ii) Evaluation

We compute the error rate (1 - accuracy) on the test set, separately for known words, unknown words, and overall.

In [3]:
from tagger import evaluate_baseline

total_err, known_err, unk_err = evaluate_baseline(word_to_tag, test_sents)

print(
    f"Vocabulary size (training): {len(word_to_tag)}\n"
    f"Known word error rate:      {known_err:.4f}\n"
    f"Unknown word error rate:    {unk_err:.4f}\n"
    f"Total error rate:           {total_err:.4f}"
)

Vocabulary size (training): 13574
Known word error rate:      0.0704
Unknown word error rate:    0.7504
Total error rate:           0.1481


## (c) Bigram HMM tagger

### (i) Training

We estimate transition probabilities q(t | prev_tag) and emission probabilities e(word | tag) from the training set using maximum likelihood. Each sentence is bracketed by START and STOP symbols so boundary transitions are counted.

In [4]:
from tagger import train_bigram_hmm, make_mle_emission

q, tags, vocab, word_tag_counts, emit_tag_counts = train_bigram_hmm(train_sents)
e_mle = make_mle_emission(word_tag_counts, emit_tag_counts)

print(
    f"Number of distinct tags:      {len(tags)}\n"
    f"Vocabulary size:              {len(vocab)}\n"
    f"Number of transition entries: {len(q)}\n"
    f"Number of emission entries:   {len(word_tag_counts)}"
)

Number of distinct tags:      98
Vocabulary size:              13574
Number of transition entries: 2331
Number of emission entries:   14660


### (ii) and (iii) Viterbi algorithm and evaluation

We implement Viterbi for the bigram HMM, working in log space to avoid underflow. Because pure MLE gives unknown words probability 0, we pin them to the tag "NN" here. We then run the tagger on every test sentence and compute error rates to compare with the baseline from (b).

In [5]:
from tagger import viterbi_bigram, evaluate_tagger

def predict_bigram_mle(sent_words):
    return viterbi_bigram(sent_words, q, e_mle, tags, vocab, force_unk_tag="NN")

total_err, known_err, unk_err = evaluate_tagger(predict_bigram_mle, test_sents, vocab)

print(
    f"Bigram HMM (MLE):\n"
    f"  Known word error rate:   {known_err:.4f}\n"
    f"  Unknown word error rate: {unk_err:.4f}\n"
    f"  Total error rate:        {total_err:.4f}"
)

Bigram HMM (MLE):
  Known word error rate:   0.1874
  Unknown word error rate: 0.7504
  Total error rate:        0.2517


 ### Comparison of (b) and (c)

  | Method | Known err | Unknown err | Total err |
  |---|---|---|---|
  | (b) Most-likely-tag baseline | 0.0704 | 0.7504 | 0.1481 |
  | (c) Bigram HMM (MLE), unknown pinned to NN | 0.1874 | 0.7504 | 0.2517 |

  The unknown-word error rate is identical in (b) and (c) because both methods predict "NN" for every unknown word: (b) by explicit fallback, (c) because pure MLE assigns probability 0 to unseen words, so `force_unk_tag="NN"` is needed to
  make Viterbi produce any output and it ends up pinning every unknown to NN. This is a direct result of the fact we chose the arbitrary tag to be NN as in section (b). 

  The bigram HMM with pure MLE in (c) underperforms even the baseline (b) on known words: with no smoothing, many tag bigrams and (word, tag) pairs have probability 0, which kills entire Viterbi paths and forces the algorithm into locally
  suboptimal choices. The baseline benefits from exact per-word lookup on the ~89% of test tokens that are known, which the HMM cannot match without smoothing.




## (d) Add-one smoothing

### (i) and (ii) Smoothed emissions and evaluation

We recompute emission probabilities using Add-one (Laplace) smoothing: e(w | t) = (count(w, t) + 1) / (count(t) + |V|). Transitions are unchanged. Because every (word, tag) pair now has nonzero probability, unknown words no longer need to be pinned to "NN", Viterbi can choose any tag for them. We then re-run Viterbi on the test set and compare error rates to (b) and (c).

In [6]:
from tagger import make_addone_emission

e_addone = make_addone_emission(word_tag_counts, emit_tag_counts, vocab)

def predict_bigram_addone(sent_words):
    return viterbi_bigram(sent_words, q, e_addone, tags, vocab, force_unk_tag=None)

total_err, known_err, unk_err = evaluate_tagger(predict_bigram_addone, test_sents, vocab)

print(
    f"Bigram HMM (Add-one smoothing):\n"
    f"  Known word error rate:   {known_err:.4f}\n"
    f"  Unknown word error rate: {unk_err:.4f}\n"
    f"  Total error rate:        {total_err:.4f}"
)

Bigram HMM (Add-one smoothing):
  Known word error rate:   0.1461
  Unknown word error rate: 0.7147
  Total error rate:        0.2110


### Comparison of (b), (c), and (d)

  | Method | Known err | Unknown err | Total err |
  |---|---|---|---|
  | (b) Most-likely-tag baseline | 0.0704 | 0.7504 | 0.1481 |
  | (c) Bigram HMM (MLE), unknown pinned to NN | 0.1874 | 0.7504 | 0.2517 |
  | (d) Bigram HMM (Add-one smoothing) | 0.1461 | 0.7147 | 0.2110 |

  Add-one smoothing makes every (word, tag) pair nonzero, so the `force_unk_tag="NN"` pin from (c) is no longer needed: Viterbi can choose any tag for an unknown word based on the smoothed emissions and transitions. This is why the
  unknown-word error rate finally moves off 0.7504, dropping to 0.7147, the first time the HMM does anything other than blanket-tagging unknowns as NN.

  Smoothing also helps on known words: with no zero-probability emissions to kill Viterbi paths, the algorithm avoids the locally suboptimal choices that hurt (c), and known-word error drops from 0.1874 to 0.1461. The HMM still does not beat
   the baseline overall, however: assigning a uniform pseudo-count of 1 to every unseen (word, tag) pair washes out the true distribution, and the baseline's exact per-word lookup on the ~89% of test tokens that are known remains hard to
  match.


## Designing pseudo-words: data inspection

Before designing pseudo-word categories for (e), we look at the actual low-frequency training words and unknown test words. The categories should cover the surface features (digits, capitalization, suffixes, hyphenation, currency symbols, etc.) that show up here.

In [7]:
from tagger import diagnose_rare_and_unknown

diag = diagnose_rare_and_unknown(train_sents, test_sents, low_freq_threshold=5)

print(
    f"Low-frequency training words (count < {diag['low_freq_threshold']}): {diag['num_low_freq']}\n"
    f"Sample of low-frequency training words:\n  {diag['low_freq_sample']}\n\n"
    f"Unknown test words (not in training vocab): {diag['num_unknown_test']}\n"
    f"Sample of unknown test words:\n  {diag['unknown_test_sample']}"
)

Low-frequency training words (count < 5): 11177
Sample of low-frequency training words:
  ['$1,000', '$1,000,000,000', '$1,500', '$1,500,000', '$1,600', '$1,800', '$1.1', '$1.4', '$1.5', '$1.80', '$10', '$102,285,000', '$109', '$11.50', '$115,000', '$12', '$12,192,865', '$12,500', '$12.50', '$12.7', '$120', '$125', '$135', '$14', '$15', '$15,000', '$15,000,000', '$150', '$157,460', '$16', '$17', '$17,000', '$17.8', '$172,000', '$172,400', '$18', '$18.2', '$18.9', '$2', '$2,000']

Unknown test words (not in training vocab): 820
Sample of unknown test words:
  ['$10,000-per-year', '$100,000', '$139.3', '$16,000', '$2.80', '$300', '$32,000', '$46.7', '$5.2', '$754', '0', '1,509', '12-month', '121', '124', '12:50', '13.5', '15,000', '16-22', '160,000', '173', '175', '1851', '186', '187-mile', '1885', '19,000', '1908', '1965', '1:35', '2,100', '21-year', '224-170', '225,000', '25,000', '25,000,000', '25,000-man', '36-year-old', '3:57', '4:18']


## (e) Pseudo-words

### (i) Pseudo-word design

Based on the diagnostic above, we define pseudo-word categories covering the surface patterns of the unknown and low-frequency words: dollar amounts, times with colons, numbers with commas or periods, year-like four-digit numbers, hyphenated compounds, capitalized proper nouns, all-caps acronyms, single-letter initials, and a few catch-all categories. In training we replace any word with count below the threshold (5) by its pseudo-word; at test time we replace any word not in the augmented training vocabulary.

In [8]:
from tagger import (
    apply_pseudo_words_to_training,
    apply_pseudo_words_to_test,
    pseudo_word,
    build_confusion_matrix,
)

train_sents_pw, vocab_pw = apply_pseudo_words_to_training(train_sents, threshold=5)
test_sents_pw = apply_pseudo_words_to_test(test_sents, vocab_pw)

print(
    f"Augmented training vocabulary size: {len(vocab_pw)}\n"
    f"Sample of training sentence after pseudo-word replacement:\n"
    f"  {train_sents_pw[0][:8]}"
)

Augmented training vocabulary size: 2410
Sample of training sentence after pseudo-word replacement:
  [('The', 'AT'), ('Fulton', 'NP'), ('County', 'NN'), ('Grand', 'JJ'), ('initCap', 'NN'), ('said', 'VBD'), ('Friday', 'NR'), ('an', 'AT')]


### (ii) MLE evaluation with pseudo-words

We retrain the bigram HMM with MLE on the pseudo-word-augmented training set and run Viterbi on the augmented test set. The original test sentences (not the pseudo-word-replaced ones) are still used for error reporting so that known/unknown breakdowns remain comparable with (b), (c), (d).

In [9]:
q_pw, tags_pw, vocab_pw2, wtc_pw, etc_pw = train_bigram_hmm(train_sents_pw)
e_mle_pw = make_mle_emission(wtc_pw, etc_pw)

def predict_pw_mle(sent_words):
    sent_in = [w if w in vocab_pw2 else pseudo_word(w) for w in sent_words]
    return viterbi_bigram(sent_in, q_pw, e_mle_pw, tags_pw, vocab_pw2, force_unk_tag="NN")

total_err, known_err, unk_err = evaluate_tagger(predict_pw_mle, test_sents, vocab)

print(
    f"Pseudo-words + MLE:\n"
    f"  Known word error rate:   {known_err:.4f}\n"
    f"  Unknown word error rate: {unk_err:.4f}\n"
    f"  Total error rate:        {total_err:.4f}"
)

Pseudo-words + MLE:
  Known word error rate:   0.2085
  Unknown word error rate: 0.5497
  Total error rate:        0.2475


### Comparison of (b), (c), (d), and (e)(ii)

  | Method | Known err | Unknown err | Total err |
  |---|---|---|---|
  | (b) Most-likely-tag baseline | 0.0704 | 0.7504 | 0.1481 |
  | (c) Bigram HMM (MLE), unknown pinned to NN | 0.1874 | 0.7504 | 0.2517 |
  | (d) Bigram HMM (Add-one smoothing) | 0.1461 | 0.7147 | 0.2110 |
  | (e)(ii) Pseudo-words + MLE, unknown pinned to NN | 0.2085 | 0.5497 | 0.2475 |

  Pseudo-words deliver a large drop in unknown-word error compared to every earlier method: 0.7504 in (b) and (c), 0.7147 in (d), now 0.5497. Categories like `dollarAmount`, `fourDigitNum`, `containsDigitAndComma`, and `initCap` give the
  model a real chance on tokens that previously had no useful evidence: a token like `Holmes` is no longer a blank `NN` default but is mapped to `initCap`, which the HMM has learned to tag mostly as `NP`.

  Known-word error, however, is slightly worse than in (c) and (d): 0.2085 vs 0.1874 / 0.1461. This is because the pseudo-word fold also affects "known" tokens at evaluation time: roughly 14% of test tokens that are in the original training
  vocabulary are low-frequency words that got replaced by coarse category labels in training, so at test time Viterbi sees the category instead of the exact word for them. That gives up the precise emission signal those tokens would have
  otherwise carried.
  
  Total error is close to (c) and slightly worse than (d): the gain on the ~11% of test tokens that are unknown is partly washed out by the small regression on the ~89% that are known. 



### (iii) Add-one smoothing with pseudo-words and confusion matrix

Same setup, with Add-one smoothing on the emissions. We also build a confusion matrix over the test set and report the most frequent confusion pairs (true tag, predicted tag).

In [10]:
e_addone_pw = make_addone_emission(wtc_pw, etc_pw, vocab_pw2)

def predict_pw_addone(sent_words):
    sent_in = [w if w in vocab_pw2 else pseudo_word(w) for w in sent_words]
    return viterbi_bigram(sent_in, q_pw, e_addone_pw, tags_pw, vocab_pw2, force_unk_tag=None)

total_err, known_err, unk_err = evaluate_tagger(predict_pw_addone, test_sents, vocab)
cm = build_confusion_matrix(predict_pw_addone, test_sents)
top_errors = [((g, p), c) for (g, p), c in cm.most_common() if g != p][:10]

lines = [
    f"Pseudo-words + Add-one smoothing:",
    f"  Known word error rate:   {known_err:.4f}",
    f"  Unknown word error rate: {unk_err:.4f}",
    f"  Total error rate:        {total_err:.4f}",
    "",
    f"Top 10 confusion pairs (true -> predicted):",
]
for (g, p), c in top_errors:
    lines.append(f"  {g:>6} -> {p:<6}  count = {c}")

print(chr(10).join(lines))

Pseudo-words + Add-one smoothing:
  Known word error rate:   0.1484
  Unknown word error rate: 0.5358
  Total error rate:        0.1927

Top 10 confusion pairs (true -> predicted):
     NNS -> NN      count = 266
      NN -> NP      count = 103
      NN -> JJ      count = 87
      JJ -> NN      count = 69
      NP -> NN      count = 56
      JJ -> NP      count = 46
     VBG -> NN      count = 45
      VB -> NN      count = 40
     VBD -> NN      count = 39
      RB -> NN      count = 38


### Comparison of (b), (c), (d), (e)(ii), and (e)(iii)

  | Method | Known err | Unknown err | Total err |
  |---|---|---|---|
  | (b) Most-likely-tag baseline | 0.0704 | 0.7504 | 0.1481 |
  | (c) Bigram HMM (MLE), unknown pinned to NN | 0.1874 | 0.7504 | 0.2517 |
  | (d) Bigram HMM (Add-one smoothing) | 0.1461 | 0.7147 | 0.2110 |
  | (e)(ii) Pseudo-words + MLE, unknown pinned to NN | 0.2085 | 0.5497 | 0.2475 |
  | (e)(iii) Pseudo-words + Add-one smoothing | 0.1484 | 0.5358 | 0.1927 |

  The combination of pseudo-words with Add-one smoothing gives the best total error rate of any HMM variant (0.1927) and the best unknown-word error rate overall (0.5358). The unknown-word error rate drops from 0.7504 in (b) and (c) all the
  way to 0.5358 here, a gain of more than 21 percentage points: pseudo-words give the model real surface evidence about unknown tokens (digits, capitalization, currency symbols, hyphenated compounds) and Add-one smoothing lets Viterbi
  actually use that evidence instead of collapsing to "NN" by default.
  
  On known words, (e)(iii) at 0.1484 recovers almost all of the ground lost in (e)(ii) (0.2085) and matches (d) (0.1461): the same logic that helps unknowns (every (word, tag) pair nonzero, so Viterbi never has to pick from a dead path) also
   fixes the known-word regression caused by pseudo-word folding.

  The baseline (b) still has the lowest total error rate, because it benefits from exact per-word lookup on the ~89% of test tokens that are known, which the HMM cannot fully match. But (e)(iii) is the first method that beats the baseline on
   unknown words by a wide margin while staying competitive on known words, and it is the only variant where the HMM's structural advantage (context via transitions) actually pays off across all token types.

  ### Investigation of the most frequent errors

  The top confusion pairs cluster around three patterns:

  1. **NNS → NN (266 errors), the dominant single mistake.** Plural nouns are being collapsed to singular. The pseudo-word categories do not encode a "plural-s suffix" signal, so when a plural noun is low-frequency in training and gets
  folded into `lowercase`, Viterbi has no way to recover the NNS reading and defaults to the much more common NN under the same surface form.

  2. **NN ↔ NP / NN ↔ JJ symmetric confusions (103 + 87 + 69 + 56 + 46 = 361 errors combined).** These are the classic ambiguities in English POS tagging: capitalized common nouns vs proper nouns (e.g., `Mr.` in headlines), adjective vs noun
   usages of the same surface form (e.g., `Executive` as JJ in "Executive Committee" vs NN in "the executive"). The pseudo-word `initCap` lumps proper nouns and title-cased common nouns into one bin, which the HMM can only disambiguate
  through transitions, and transitions are unsmoothed.
  
  3. **Open-class words → NN (VBG 45, VB 40, VBD 39, RB 38).** Verbs and adverbs collapsing to NN. This is the structural pull of Add-one smoothing: assigning uniform pseudo-count 1 to every unseen (word, tag) pair gives the most frequent
  tag (NN) a strong prior advantage whenever the true tag is rare or the transition feeding it is unseen. Whenever Viterbi has to choose under sparse evidence, NN wins.
